# Time of Emergence — Three Variables, Three Methods
## G6-1.5K-HiLLA vs SSP2-4.5  |  MIROC-ES2H

Three complementary detection methods applied to three climate variables:

| Row | Variable | SAI signal | Test direction |
|---|---|---|---|
| 1 | GMSAT | SAI cools | HiLLA < SSP2-4.5 |
| 2 | Arctic September SIA | SAI preserves ice | HiLLA > SSP2-4.5 |
| 3 | SA Monsoon JJA precip (5–40N, 60–100E; Nalam et al. 2018) | SAI weakens monsoon | HiLLA < SSP2-4.5 |

All anomalies relative to the 2020–2039 SSP2-4.5 ensemble mean of each variable.

In [1]:
import sys
import fsspec
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from scipy import stats

matplotlib.rcParams.update({'font.size': 12})
sys.path.append('../shared/Code_examples/G6-1.5K_example/')
from Utils import weighted_annual_resample

SAI_START  = 2035
ANOM_REF   = (2020, 2039)
PLOT_START = 2020
PLOT_END   = 2084
K_CONSEC   = 5
CI_LEVEL   = 0.95
ALPHA      = 0.05
MIN_N      = 5
Y_MIN, Y_MAX = 2, 40
R_EARTH    = 6.371e6  # m

MON_LAT = (5, 40)    # Nalam et al. 2018 SA monsoon box
MON_LON = (60, 100)

BLUE       = '#2171B5'
BLUE_LIGHT = '#BDD7E7'
GRAY       = '#636363'
GRAY_LIGHT = '#D9D9D9'

In [2]:
_MIROC_AMON = 's3://reflective-persistent-prod-large/MIROC-ES2H/G6-1.5K-HiLLA/Amon'
_MIROC_OMON = 's3://reflective-persistent-prod-large/MIROC-ES2H/G6-1.5K-HiLLA/Omon'
N_MEM = 10

gmsat_HiLLA_files  = [f'{_MIROC_AMON}/tas_G6-1.5K-HiLLA_r{i:02d}.nc'   for i in range(1, N_MEM + 1)]
gmsat_ssp245_files = [f'{_MIROC_AMON}/tas_baseline_r{i:02d}.nc'          for i in range(1, N_MEM + 1)]
sia_HiLLA_files    = [f'{_MIROC_OMON}/siconc_G6-1.5K-HiLLA_r{i:02d}.nc' for i in range(1, N_MEM + 1)]
sia_ssp245_files   = [f'{_MIROC_OMON}/siconc_baseline_r{i:02d}.nc'       for i in range(1, N_MEM + 1)]
pr_HiLLA_files     = [f'{_MIROC_AMON}/pr_G6-1.5K-HiLLA_r{i:02d}.nc'    for i in range(1, N_MEM + 1)]
pr_ssp245_files    = [f'{_MIROC_AMON}/pr_baseline_r{i:02d}.nc'           for i in range(1, N_MEM + 1)]
print(f'Paths: {N_MEM} members x 3 variables x 2 scenarios = {N_MEM * 6} files.')

Paths: 10 members x 3 variables x 2 scenarios = 60 files.


In [3]:
def open_nc(path):
    with fsspec.open(path, mode='rb') as f:
        return xr.open_dataset(f, engine='h5netcdf', chunks={}).load()


def global_annual_mean(ds, var='tas', lat_nm='lat'):
    da = ds[var]
    if float(da[lat_nm][0]) > float(da[lat_nm][-1]):
        da = da.sortby(lat_nm)
    weights = np.cos(np.deg2rad(da[lat_nm]))
    weights /= weights.sum()
    gmean  = (da * weights).sum(lat_nm).mean('lon')
    annual = weighted_annual_resample(gmean.to_dataset(name=var), var=var)[var]
    return annual.assign_coords(time=annual.time.dt.year.values)


def read_miroc_ice(file_list):
    ds_list = []
    for f in file_list:
        with fsspec.open(f, mode='rb') as fobj:
            ds = xr.open_dataset(fobj, engine='h5netcdf', chunks={}).load()
        extra = [d for d in ds.dims if d not in ('lon', 'time') and d != 'lat']
        if extra:
            ds = ds.rename({extra[0]: 'lat'})
        ice_var = next(
            (v for v in ds.data_vars
             if 'ice' in v.lower() or 'conc' in v.lower() or v == 'SeaIce'), None
        )
        if ice_var and ice_var != 'siconc':
            ds = ds.rename({ice_var: 'siconc'})
        if 'siconc' in ds and float(ds['siconc'].max()) <= 1.5:
            ds['siconc'] = ds['siconc'] * 100.0
        ds_list.append(ds)
    return ds_list


def cell_area_m2(lat, lon):
    dlat    = np.deg2rad(abs(float(lat[1] - lat[0])))
    dlon    = np.deg2rad(abs(float(lon[1] - lon[0])))
    cos_lat = np.cos(np.deg2rad(lat))
    return (R_EARTH**2 * cos_lat * dlat * dlon).expand_dims({'lon': len(lon)}, axis=1)


def september_sia(ds_list, lat_min=0.0):
    result = []
    for ds in ds_list:
        da     = ds['siconc']
        lat_nm = 'lat' if 'lat' in da.dims else 'latitude'
        lon_nm = 'lon' if 'lon' in da.dims else 'longitude'
        if float(da[lat_nm][0]) > float(da[lat_nm][-1]):
            da = da.isel(**{lat_nm: slice(None, None, -1)})
        da_nh = da.sel(**{lat_nm: slice(lat_min, 90)})
        area  = cell_area_m2(da_nh[lat_nm], da_nh[lon_nm])
        sia   = ((da_nh / 100.0) * area).sum([lat_nm, lon_nm]) / 1e12
        sep   = sia.sel(time=sia.time.dt.month == 9)
        result.append(sep.assign_coords(time=sep.time.dt.year.values.astype(int)))
    return result


def jja_monsoon_mean(ds, var='pr', lat_nm='lat', lon_nm='lon'):
    da = ds[var]
    if float(da[lat_nm][0]) > float(da[lat_nm][-1]):
        da = da.sortby(lat_nm)
    da_box  = da.sel(**{lat_nm: slice(*MON_LAT), lon_nm: slice(*MON_LON)})
    weights = np.cos(np.deg2rad(da_box[lat_nm]))
    weights = weights / weights.sum()
    sp_mean = (da_box * weights).sum(lat_nm).mean(lon_nm)
    jja     = sp_mean.sel(time=sp_mean.time.dt.month.isin([6, 7, 8]))
    annual  = jja.groupby('time.year').mean('time') * 86400.0   # kg/m2/s -> mm/day
    return annual.rename({'year': 'time'})


print('Loading functions defined.')

Loading functions defined.


In [ ]:
print('=== Loading GMSAT (tas) ===')
gmsat_HiLLA_ds  = [open_nc(f) for f in gmsat_HiLLA_files]
gmsat_ssp245_ds = [open_nc(f) for f in gmsat_ssp245_files]
print(f'  HiLLA : {int(gmsat_HiLLA_ds[0].time.dt.year.min())}–{int(gmsat_HiLLA_ds[0].time.dt.year.max())}')
print(f'  SSP245: {int(gmsat_ssp245_ds[0].time.dt.year.min())}–{int(gmsat_ssp245_ds[0].time.dt.year.max())}')

print('\n=== Loading Sea Ice (siconc) ===')
sia_HiLLA_ds  = read_miroc_ice(sia_HiLLA_files)
sia_ssp245_ds = read_miroc_ice(sia_ssp245_files)
print(f'  HiLLA : {len(sia_HiLLA_ds)} members loaded')
print(f'  SSP245: {len(sia_ssp245_ds)} members loaded')

print('\n=== Loading Monsoon (pr) ===')
pr_HiLLA_ds  = [open_nc(f) for f in pr_HiLLA_files]
pr_ssp245_ds = [open_nc(f) for f in pr_ssp245_files]
print(f'  HiLLA : {int(pr_HiLLA_ds[0].time.dt.year.min())}–{int(pr_HiLLA_ds[0].time.dt.year.max())}')
print(f'  SSP245: {int(pr_ssp245_ds[0].time.dt.year.min())}–{int(pr_ssp245_ds[0].time.dt.year.max())}')

=== Loading GMSAT (tas) ===


In [ ]:
print('Computing GMSAT annual means...')
gmsat_HiLLA_ts  = [global_annual_mean(ds) for ds in gmsat_HiLLA_ds]
gmsat_ssp245_ts = [global_annual_mean(ds) for ds in gmsat_ssp245_ds]

print('Computing September SIA...')
sia_HiLLA_ts  = september_sia(sia_HiLLA_ds)
sia_ssp245_ts = september_sia(sia_ssp245_ds)

print('Computing JJA monsoon means...')
pr_HiLLA_ts  = [jja_monsoon_mean(ds) for ds in pr_HiLLA_ds]
pr_ssp245_ts = [jja_monsoon_mean(ds) for ds in pr_ssp245_ds]


def ens_mean_ref(ts_list, yr_start=ANOM_REF[0], yr_end=ANOM_REF[1]):
    return float(np.mean([
        float(ts.sel(time=slice(yr_start, yr_end)).mean())
        for ts in ts_list
    ]))


ref_gmsat = ens_mean_ref(gmsat_ssp245_ts)
ref_sia   = ens_mean_ref(sia_ssp245_ts)
ref_pr    = ens_mean_ref(pr_ssp245_ts)
print(f'Anomaly refs — GMSAT: {ref_gmsat:.4f} K | SIA: {ref_sia:.3f} Mkm2 | PR: {ref_pr:.4f} mm/day')

gmsat_HiLLA_anom  = [ts - ref_gmsat for ts in gmsat_HiLLA_ts]
gmsat_ssp245_anom = [ts - ref_gmsat for ts in gmsat_ssp245_ts]
sia_HiLLA_anom    = [ts - ref_sia   for ts in sia_HiLLA_ts]
sia_ssp245_anom   = [ts - ref_sia   for ts in sia_ssp245_ts]
pr_HiLLA_anom     = [ts - ref_pr    for ts in pr_HiLLA_ts]
pr_ssp245_anom    = [ts - ref_pr    for ts in pr_ssp245_ts]


def stack_ens(ts_list, yr_start=PLOT_START, yr_end=PLOT_END):
    common = sorted(set.intersection(*[set(ts.time.values.tolist()) for ts in ts_list]))
    common = [y for y in common if yr_start <= y <= yr_end]
    return (np.stack([ts.sel(time=common).values for ts in ts_list]),
            np.array(common, dtype=int))


gmsat_h_arr, gmsat_h_yrs = stack_ens(gmsat_HiLLA_anom)
gmsat_s_arr, gmsat_s_yrs = stack_ens(gmsat_ssp245_anom)
sia_h_arr,   sia_h_yrs   = stack_ens(sia_HiLLA_anom)
sia_s_arr,   sia_s_yrs   = stack_ens(sia_ssp245_anom)
pr_h_arr,    pr_h_yrs    = stack_ens(pr_HiLLA_anom)
pr_s_arr,    pr_s_yrs    = stack_ens(pr_ssp245_anom)
print(f'Stacked: GMSAT H{gmsat_h_arr.shape}/S{gmsat_s_arr.shape} | '
      f'SIA H{sia_h_arr.shape}/S{sia_s_arr.shape} | '
      f'PR H{pr_h_arr.shape}/S{pr_s_arr.shape}')

In [ ]:
y_vals     = np.arange(Y_MIN, Y_MAX + 1)
PRED_YEARS = np.arange(SAI_START, PLOT_END + 1)


def perfect_model_toe(hilla_list, ssp245_list, alternative='less',
                      start_year=SAI_START, y_vals=y_vals, alpha=ALPHA):
    p_arr = np.full(len(y_vals), np.nan)
    for i, Y in enumerate(y_vals):
        h = [float(ts.sel(time=slice(start_year, start_year + int(Y) - 1)).mean())
             for ts in hilla_list]
        s = [float(ts.sel(time=slice(start_year, start_year + int(Y) - 1)).mean())
             for ts in ssp245_list]
        _, p = stats.ttest_ind(h, s, equal_var=False, alternative=alternative)
        p_arr[i] = p
    first = np.where(p_arr < alpha)[0]
    toe  = int(start_year + y_vals[first[0]] - 1) if len(first) else None
    wlen = int(y_vals[first[0]]) if len(first) else None
    return p_arr, toe, wlen


def cumulative_ttest_toe(hilla_ts, ssp245_ts, alternative='less',
                         start_year=SAI_START, alpha=ALPHA, min_n=MIN_N):
    h_yrs = hilla_ts.time.values.astype(int);  h_vals = hilla_ts.values
    s_yrs = ssp245_ts.time.values.astype(int); s_vals = ssp245_ts.values
    h_yrs, h_vals = h_yrs[h_yrs >= start_year], h_vals[h_yrs >= start_year]
    s_yrs, s_vals = s_yrs[s_yrs >= start_year], s_vals[s_yrs >= start_year]
    end_year = int(min(h_yrs.max(), s_yrs.max()))
    p_vals = {}; toe = None
    for Y in range(start_year, end_year + 1):
        h_win = h_vals[h_yrs <= Y]
        s_win = s_vals[s_yrs <= Y]
        if min(len(h_win), len(s_win)) < min_n:
            p_vals[Y] = np.nan; continue
        _, p = stats.ttest_ind(h_win, s_win, equal_var=False, alternative=alternative)
        p_vals[Y] = p
        if toe is None and p < alpha:
            toe = Y
    return toe, p_vals


def model_cumttest_toe(hilla_list, ssp245_list, alternative='less', **kwargs):
    mem_toes = []; pvals_all = []
    for h, s in zip(hilla_list, ssp245_list):
        t, pv = cumulative_ttest_toe(h, s, alternative=alternative, **kwargs)
        mem_toes.append(t); pvals_all.append(pv)
    all_yrs = sorted(set().union(*[set(pv.keys()) for pv in pvals_all]))
    m_toe   = next((Y for Y in all_yrs
                    if all(pv.get(Y, np.nan) < ALPHA for pv in pvals_all)), None)
    return mem_toes, m_toe, pvals_all


def ar1_forecast(obs_ts, pred_years, ci=CI_LEVEL):
    x_fit = obs_ts.time.values.astype(float)
    y_fit = obs_ts.values.astype(float)
    slope, intercept, *_ = stats.linregress(x_fit, y_fit)
    resid    = y_fit - (slope * x_fit + intercept)
    phi      = np.dot(resid[1:], resid[:-1]) / np.dot(resid[:-1], resid[:-1])
    innov    = resid[1:] - phi * resid[:-1]
    sigma    = np.sqrt(np.sum(innov**2) / (len(innov) - 1))
    eps_last = resid[-1]
    h        = pred_years.astype(float) - x_fit[-1]
    mu       = slope * pred_years + intercept + phi**h * eps_last
    z        = stats.norm.ppf((1 + ci) / 2)
    pi_half  = z * sigma * np.sqrt((1 - phi**(2*h)) / max(1 - phi**2, 1e-8))
    return mu, mu - pi_half, mu + pi_half


def ar1_toe(hilla_ts, pred_years, pi_bound, start_year=SAI_START,
            K=K_CONSEC, direction='below'):
    h_yrs  = hilla_ts.time.values
    common = np.intersect1d(h_yrs[h_yrs >= start_year], pred_years)
    if len(common) < K:
        return None
    h_vals = hilla_ts.sel(time=common).values
    bound  = pi_bound[np.searchsorted(pred_years, common)]
    exceed = (h_vals < bound) if direction == 'below' else (h_vals > bound)
    for i in range(len(exceed) - K + 1):
        if exceed[i: i + K].all():
            return int(common[i])
    return None


def plot_ens(ax, arr, years, color, light, label, lw=2.0):
    mean = arr.mean(axis=0)
    p05  = np.percentile(arr,  5, axis=0)
    p95  = np.percentile(arr, 95, axis=0)
    ax.fill_between(years, p05, p95, color=light, alpha=0.6, zorder=1)
    ax.plot(years, mean, color=color, lw=lw, label=label, zorder=2)


def mark_toe(ax, toe, color, note=''):
    ax.axvline(toe, color=color, lw=2.0, ls=':', zorder=5)
    label = f'ToE={toe}' if not note else f'ToE={toe} ({note})'
    ax.text(toe + 0.4, 0.97, label,
            transform=ax.get_xaxis_transform(),
            color=color, fontsize=10.5, va='top', fontweight='bold')


def plot_members(ax, ts_list, color, yr_start=PLOT_START, yr_end=PLOT_END,
                 label_ex='', label_oth='', ex_idx=0):
    for i, ts in enumerate(ts_list):
        yrs  = ts.time.values
        mask = (yrs >= yr_start) & (yrs <= yr_end)
        if i == ex_idx:
            ax.plot(yrs[mask], ts.values[mask], color=color,
                    lw=2.5, ls=':', alpha=1.0, zorder=4, label=label_ex)
        else:
            ax.plot(yrs[mask], ts.values[mask], color=color,
                    lw=0.9, ls='-', alpha=0.35, zorder=3,
                    label=label_oth if i == 1 else '_')


print('Analysis functions defined.')

In [ ]:
VARS = [
    {
        'name':   'GMSAT',
        'ylabel': 'GMSAT anomaly (K)',
        'h_list': gmsat_HiLLA_anom,  's_list': gmsat_ssp245_anom,
        'h_arr':  gmsat_h_arr,        'h_yrs':  gmsat_h_yrs,
        's_arr':  gmsat_s_arr,        's_yrs':  gmsat_s_yrs,
        'alt':    'less',  'ar1_dir': 'below',
    },
    {
        'name':   'Arctic Sept SIA',
        'ylabel': 'SIA anomaly (10^6 km^2)',
        'h_list': sia_HiLLA_anom,  's_list': sia_ssp245_anom,
        'h_arr':  sia_h_arr,        'h_yrs':  sia_h_yrs,
        's_arr':  sia_s_arr,        's_yrs':  sia_s_yrs,
        'alt':    'greater', 'ar1_dir': 'above',
    },
    {
        'name':   'SA Monsoon JJA',
        'ylabel': 'Monsoon anomaly (mm/day)',
        'h_list': pr_HiLLA_anom,  's_list': pr_ssp245_anom,
        'h_arr':  pr_h_arr,        'h_yrs':  pr_h_yrs,
        's_arr':  pr_s_arr,        's_yrs':  pr_s_yrs,
        'alt':    'less',  'ar1_dir': 'below',
    },
]

for cfg in VARS:
    print(f"\n=== {cfg['name']} ===")
    alt     = cfg['alt']
    ar1_dir = cfg['ar1_dir']

    # Method 1: perfect-model window t-test
    _, m1_toe, m1_wlen = perfect_model_toe(cfg['h_list'], cfg['s_list'], alternative=alt)
    cfg['m1_toe'] = m1_toe;  cfg['m1_wlen'] = m1_wlen
    print(f'  M1 (perfect-model) : ToE={m1_toe}, window={m1_wlen} yr')

    # Method 2: cumulative t-test (model ToE = all members pass)
    m2_mem, m2_toe, m2_pvs = model_cumttest_toe(cfg['h_list'], cfg['s_list'], alternative=alt)
    cfg['m2_mem'] = m2_mem;  cfg['m2_toe'] = m2_toe
    print(f'  M2 (cumul. t-test) : members={m2_mem}')
    print(f'                       model ToE={m2_toe}')

    # Method 3: obs-only AR(1) on r01
    obs_r01    = cfg['s_list'][0].sel(time=slice(2020, SAI_START - 1))
    mu, lo, hi = ar1_forecast(obs_r01, PRED_YEARS)
    pi_bd      = lo if ar1_dir == 'below' else hi
    m3_toe     = ar1_toe(cfg['h_list'][0], PRED_YEARS, pi_bd, direction=ar1_dir)
    cfg['ar1_mu'] = mu;  cfg['ar1_lo'] = lo;  cfg['ar1_hi'] = hi
    cfg['m3_toe'] = m3_toe
    print(f'  M3 (obs-only AR1)  : ToE={m3_toe}')

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 14), sharey='row')
fig.subplots_adjust(hspace=0.40, wspace=0.06)

col0_title = 'Method 1: Perfect-model\nEnsemble window t-test (n=10 vs n=10)'
col1_title = 'Method 2: SSP2-4.5 counterfactual\nCumulative t-test (per-member, all pass)'
col2_title = 'Method 3: Obs-only\nOLS + AR(1) counterfactual (r01)'

for row, cfg in enumerate(VARS):
    h_arr, h_yrs = cfg['h_arr'], cfg['h_yrs']
    s_arr, s_yrs = cfg['s_arr'], cfg['s_yrs']

    # ── Col 0: Method 1 ─────────────────────────────────────────────────────
    ax = axes[row, 0]
    plot_ens(ax, s_arr[:, s_yrs >= PLOT_START], s_yrs[s_yrs >= PLOT_START],
             GRAY, GRAY_LIGHT, 'SSP2-4.5 (n=10)')
    plot_ens(ax, h_arr, h_yrs, BLUE, BLUE_LIGHT, 'G6-1.5K-HiLLA (n=10)')
    ax.axvline(SAI_START, color='k', lw=1.2, ls='--', alpha=0.45, label='SAI start (2035)')
    if cfg['m1_toe']:
        wlen = cfg['m1_wlen']
        mark_toe(ax, cfg['m1_toe'], BLUE, note=f'win.={wlen}yr')
    ax.set_ylabel(cfg['name'] + '\n' + cfg['ylabel'], fontsize=9.5)
    if row == 0:
        ax.set_title(col0_title, fontsize=11, pad=7)
        ax.legend(fontsize=9, loc='upper left', framealpha=0.85)
    if row == 2:
        ax.set_xlabel('Year', fontsize=11)
    ax.set_xlim(PLOT_START, PLOT_END)
    ax.grid(ls='--', alpha=0.3)
    for sp in ('top', 'right'):
        ax.spines[sp].set_visible(False)

    # ── Col 1: Method 2 ─────────────────────────────────────────────────────
    ax = axes[row, 1]
    plot_ens(ax, s_arr[:, s_yrs >= PLOT_START], s_yrs[s_yrs >= PLOT_START],
             GRAY, GRAY_LIGHT, 'SSP2-4.5 (n=10)')
    plot_ens(ax, h_arr, h_yrs, BLUE, BLUE_LIGHT, 'G6-1.5K-HiLLA (n=10)')
    ax.axvline(SAI_START, color='k', lw=1.2, ls='--', alpha=0.45)
    for mt in cfg['m2_mem']:
        if mt is not None:
            ax.axvline(mt, color=BLUE, lw=0.8, ls='-', alpha=0.35, zorder=3)
    if cfg['m2_toe']:
        mark_toe(ax, cfg['m2_toe'], BLUE)
    ax.tick_params(labelleft=False)
    if row == 0:
        ax.set_title(col1_title, fontsize=11, pad=7)
        ax.legend(fontsize=9, loc='upper left', framealpha=0.85)
    if row == 2:
        ax.set_xlabel('Year', fontsize=11)
    ax.set_xlim(PLOT_START, PLOT_END)
    ax.grid(ls='--', alpha=0.3)
    for sp in ('top', 'right'):
        ax.spines[sp].set_visible(False)

    # ── Col 2: Method 3 ─────────────────────────────────────────────────────
    ax = axes[row, 2]
    obs_plot = cfg['s_list'][0].sel(time=slice(PLOT_START, SAI_START - 1))
    ax.plot(obs_plot.time.values, obs_plot.values,
            color=GRAY, lw=2.2, label='SSP2-4.5 r01 (pseudo-obs)', zorder=3)
    pi_pct = str(int(CI_LEVEL * 100))
    ax.fill_between(PRED_YEARS, cfg['ar1_lo'], cfg['ar1_hi'],
                    color=GRAY_LIGHT, alpha=0.7, zorder=1,
                    label='AR(1) PI (' + pi_pct + '%)')
    ax.plot(PRED_YEARS, cfg['ar1_mu'], color=GRAY, lw=1.8, ls='--', zorder=2)
    h_r01 = cfg['h_list'][0].sel(time=slice(SAI_START, PLOT_END))
    ax.plot(h_r01.time.values, h_r01.values,
            color=BLUE, lw=2.2, label='G6-1.5K-HiLLA r01', zorder=4)
    ax.axvline(SAI_START, color='k', lw=1.2, ls='--', alpha=0.45, label='SAI start (2035)')
    if cfg['m3_toe']:
        mark_toe(ax, cfg['m3_toe'], BLUE, note='K=' + str(K_CONSEC) + 'yr')
    ax.tick_params(labelleft=False)
    if row == 0:
        ax.set_title(col2_title, fontsize=11, pad=7)
        ax.legend(fontsize=9, loc='upper left', framealpha=0.85)
    if row == 2:
        ax.set_xlabel('Year', fontsize=11)
    ax.set_xlim(PLOT_START, PLOT_END)
    ax.grid(ls='--', alpha=0.3)
    for sp in ('top', 'right'):
        ax.spines[sp].set_visible(False)

fig.suptitle(
    'Time of Emergence  |  G6-1.5K-HiLLA vs SSP2-4.5  |  MIROC-ES2H',
    fontsize=13, y=1.01
)
plt.tight_layout()
plt.savefig('Figures/ToE_three_panel_v8.png', dpi=200, bbox_inches='tight')
plt.show()

print('\nSummary (first figure — ensemble, model ToE):')
for cfg in VARS:
    print(f"  {cfg['name']:20s}  "
          f"M1={cfg['m1_toe']} (win.={cfg['m1_wlen']}yr)  "
          f"M2={cfg['m2_toe']}  "
          f"M3={cfg['m3_toe']}")

---
## Second figure: individual ensemble members

Methods 2 and 3 are now applied to each of the 10 members independently.
**r01** is the highlighted example member (thick dotted line); **r02–r10** are thin solid lines.
Results are quoted as the **median** individual member ToE across the 10 members.

In [ ]:
# ── Method 2: median of individual member ToEs (already computed) ────────────
for cfg in VARS:
    valid = [t for t in cfg['m2_mem'] if t is not None]
    cfg['m2_median'] = int(np.median(valid)) if valid else None
    print(f"{cfg['name']:20s}  M2 members: {cfg['m2_mem']}  median: {cfg['m2_median']}")

# ── Method 3: per-member AR(1) ToE ────────────────────────────────────────────
print('\nMethod 3 — AR(1) per member (training 2020–2034):')
for cfg in VARS:
    ar1_dir     = cfg['ar1_dir']
    toes_all    = []
    results_all = []
    for i in range(N_MEM):
        obs_i         = cfg['s_list'][i].sel(time=slice(2020, SAI_START - 1))
        mu_i, lo_i, hi_i = ar1_forecast(obs_i, PRED_YEARS)
        pi_bd         = lo_i if ar1_dir == 'below' else hi_i
        toe_i         = ar1_toe(cfg['h_list'][i], PRED_YEARS, pi_bd, direction=ar1_dir)
        toes_all.append(toe_i)
        results_all.append((mu_i, lo_i, hi_i))
    valid = [t for t in toes_all if t is not None]
    cfg['m3_mem_toes']    = toes_all
    cfg['m3_median']      = int(np.median(valid)) if valid else None
    cfg['ar1_results_all'] = results_all
    print(f"  {cfg['name']:20s} ToEs: {toes_all}  median: {cfg['m3_median']}")

# ── Second figure: 3x3, individual member lines ───────────────────────────────
fig2, axes2 = plt.subplots(3, 3, figsize=(18, 14), sharey='row')
fig2.subplots_adjust(hspace=0.40, wspace=0.06)

col0_title = 'Method 1: Perfect-model\nEnsemble window t-test (n=10 vs n=10)'
col1_title = 'Method 2: SSP2-4.5 counterfactual\nCumulative t-test (per member, median ToE)'
col2_title = 'Method 3: Obs-only\nOLS + AR(1) (per member, median ToE)'

for row, cfg in enumerate(VARS):
    h_arr, h_yrs = cfg['h_arr'], cfg['h_yrs']
    s_arr, s_yrs = cfg['s_arr'], cfg['s_yrs']
    ar1_dir      = cfg['ar1_dir']

    # ── Col 0: Method 1 (ensemble, unchanged) ───────────────────────────────
    ax = axes2[row, 0]
    plot_ens(ax, s_arr[:, s_yrs >= PLOT_START], s_yrs[s_yrs >= PLOT_START],
             GRAY, GRAY_LIGHT, 'SSP2-4.5 (n=10)')
    plot_ens(ax, h_arr, h_yrs, BLUE, BLUE_LIGHT, 'G6-1.5K-HiLLA (n=10)')
    ax.axvline(SAI_START, color='k', lw=1.2, ls='--', alpha=0.45, label='SAI start (2035)')
    if cfg['m1_toe']:
        wlen = cfg['m1_wlen']
        mark_toe(ax, cfg['m1_toe'], BLUE, note=f'win.={wlen}yr')
    ax.set_ylabel(cfg['name'] + '\n' + cfg['ylabel'], fontsize=9.5)
    if row == 0:
        ax.set_title(col0_title, fontsize=11, pad=7)
        ax.legend(fontsize=9, loc='upper left', framealpha=0.85)
    if row == 2:
        ax.set_xlabel('Year', fontsize=11)
    ax.set_xlim(PLOT_START, PLOT_END)
    ax.grid(ls='--', alpha=0.3)
    for sp in ('top', 'right'):
        ax.spines[sp].set_visible(False)

    # ── Col 1: Method 2, individual member lines ─────────────────────────────
    ax = axes2[row, 1]
    plot_members(ax, cfg['s_list'], GRAY,
                 label_ex='SSP2-4.5 r01', label_oth='SSP2-4.5 r02-r10')
    plot_members(ax, cfg['h_list'], BLUE, yr_start=SAI_START,
                 label_ex='G6-1.5K-HiLLA r01', label_oth='G6-1.5K-HiLLA r02-r10')
    ax.axvline(SAI_START, color='k', lw=1.2, ls='--', alpha=0.45)
    for mt in cfg['m2_mem']:
        if mt is not None:
            ax.axvline(mt, color=BLUE, lw=0.8, ls='-', alpha=0.35, zorder=3)
    if cfg['m2_median']:
        mark_toe(ax, cfg['m2_median'], BLUE, note='median')
    ax.tick_params(labelleft=False)
    if row == 0:
        ax.set_title(col1_title, fontsize=11, pad=7)
        ax.legend(fontsize=9, loc='upper left', framealpha=0.85)
    if row == 2:
        ax.set_xlabel('Year', fontsize=11)
    ax.set_xlim(PLOT_START, PLOT_END)
    ax.grid(ls='--', alpha=0.3)
    for sp in ('top', 'right'):
        ax.spines[sp].set_visible(False)

    # ── Col 2: Method 3, individual member lines + r01 PI band ───────────────
    ax = axes2[row, 2]
    plot_members(ax, cfg['s_list'], GRAY, yr_end=SAI_START - 1,
                 label_ex='SSP2-4.5 r01 (pseudo-obs)', label_oth='SSP2-4.5 r02-r10')
    mu0, lo0, hi0 = cfg['ar1_results_all'][0]
    pi_pct = str(int(CI_LEVEL * 100))
    ax.fill_between(PRED_YEARS, lo0, hi0,
                    color=GRAY_LIGHT, alpha=0.7, zorder=1,
                    label='AR(1) PI r01 (' + pi_pct + '%)')
    ax.plot(PRED_YEARS, mu0, color=GRAY, lw=1.8, ls='--', zorder=2)
    plot_members(ax, cfg['h_list'], BLUE, yr_start=SAI_START,
                 label_ex='G6-1.5K-HiLLA r01', label_oth='G6-1.5K-HiLLA r02-r10')
    ax.axvline(SAI_START, color='k', lw=1.2, ls='--', alpha=0.45, label='SAI start (2035)')
    for ot in cfg['m3_mem_toes']:
        if ot is not None:
            ax.axvline(ot, color=BLUE, lw=0.8, ls='-', alpha=0.35, zorder=3)
    if cfg['m3_median']:
        mark_toe(ax, cfg['m3_median'], BLUE, note='median')
    ax.tick_params(labelleft=False)
    if row == 0:
        ax.set_title(col2_title, fontsize=11, pad=7)
        ax.legend(fontsize=9, loc='upper left', framealpha=0.85)
    if row == 2:
        ax.set_xlabel('Year', fontsize=11)
    ax.set_xlim(PLOT_START, PLOT_END)
    ax.grid(ls='--', alpha=0.3)
    for sp in ('top', 'right'):
        ax.spines[sp].set_visible(False)

fig2.suptitle(
    'Time of Emergence  |  G6-1.5K-HiLLA vs SSP2-4.5  |  MIROC-ES2H'
    '\nIndividual member lines  (r01 thick dotted, r02-r10 thin)',
    fontsize=13, y=1.01
)
plt.tight_layout()
plt.savefig('Figures/ToE_three_panel_members_v8.png', dpi=200, bbox_inches='tight')
plt.show()

print('\nSummary (second figure — individual member ToEs, median):')
for cfg in VARS:
    print(f"  {cfg['name']:20s}  "
          f"M1={cfg['m1_toe']} (win.={cfg['m1_wlen']}yr)  "
          f"M2_med={cfg['m2_median']}  "
          f"M3_med={cfg['m3_median']}")